# 手撕 LayerNorm

## 背景
LayerNorm 对每个样本独立归一化（沿特征维度），不依赖 batch size，适合 NLP。
公式：y = (x - mean) / sqrt(var + eps) * gamma + beta

## 考察点
- LayerNorm vs BatchNorm（归一化维度不同）
- 数值稳定性（eps）
- affine 参数 gamma/beta 的作用

In [ ]:
import torch
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = tuple(normalized_shape)
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x):
        # x: (..., *normalized_shape)
        dims = tuple(range(-len(self.normalized_shape), 0))
        mean = x.mean(dim=dims, keepdim=True)
        var = x.var(dim=dims, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return x_norm * self.weight + self.bias

In [ ]:
# 对比 torch.nn.LayerNorm
torch.manual_seed(42)
x = torch.randn(2, 3, 4)  # (batch, seq, dim)
ln_mine = LayerNorm(4)
ln_torch = nn.LayerNorm(4)
ln_torch.weight.data = ln_mine.weight.data.clone()
ln_torch.bias.data = ln_mine.bias.data.clone()
out_mine = ln_mine(x)
out_torch = ln_torch(x)
assert torch.allclose(out_mine, out_torch, atol=1e-6), "应与 torch 一致"
# 验证归一化性质
assert torch.allclose(out_mine[0,0].mean(), torch.tensor(0.), atol=1e-5)
assert torch.allclose(out_mine[0,0].std(unbiased=False), torch.tensor(1.), atol=1e-5)
print("✅ 与 torch.nn.LayerNorm 一致，均值≈0，标准差≈1")